# 📝 Day 4 Assignments — Middleware & DI

Work through each task. Run every cell. Use `TestClient` so it works in Colab.


In [ ]:
!pip install fastapi uvicorn httpx


## Task 1 — Pagination Dependency

**Problem:** Build a `common_pagination(skip: int = 0, limit: int = 10)` dependency. Use it in **two** endpoints: `/items` and `/users`. Each should return a payload containing the pagination dict.

**Expected output:**
```
/items?skip=2&limit=3 → {"page": {"skip": 2, "limit": 3}, "items": [...]}
/users               → {"page": {"skip": 0, "limit": 10}, "users": [...]}
```

💡 **Hint:** declare the dep parameter as `page: dict = Depends(common_pagination)` on both routes.


In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: define common_pagination

# TODO: add /items and /users using the dep

client = TestClient(app)
print(client.get("/items?skip=2&limit=3").json())
print(client.get("/users").json())


## Task 2 — Current-User Dependency

**Problem:** Build a dependency `current_user(x_user: str = Header(...))` that returns a fake user dict `{"username": x_user}`. If the header is missing, raise `HTTPException(401)`.

Protect a `/profile` endpoint with it.

**Expected output:**
```
No header  → 401
X-User=bob → {"username": "bob"}
```

💡 **Hint:** `from fastapi import Header, HTTPException`. Use `Header(default="")` and check for empty string.


In [ ]:
from fastapi import FastAPI, Depends, Header, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: define current_user dep

# TODO: add /profile protected by it

client = TestClient(app)
print(client.get("/profile").status_code)
print(client.get("/profile", headers={"X-User": "bob"}).json())


## Task 3 — Timing Middleware

**Problem:** Write a middleware that adds an `X-Process-Time` response header with the elapsed seconds. Verify with `TestClient`.

**Expected output:**
```
response.headers["X-Process-Time"]  → a small float like "0.0004"
```

💡 **Hint:** use `time.perf_counter()` before and after `await call_next(request)`.


In [ ]:
import time
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: add @app.middleware("http") that sets X-Process-Time

@app.get("/")
def root():
    return {"ok": True}

client = TestClient(app)
r = client.get("/")
print("X-Process-Time:", r.headers.get("X-Process-Time"))


## Task 4 — CORS

**Problem:** Add `CORSMiddleware` allowing only `http://localhost:3000`. Make a request with an `Origin` header and confirm the `access-control-allow-origin` header echoes back.

In a markdown cell below the code, explain in 2–3 sentences what changes when CORS is enabled vs not.


In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: add CORSMiddleware with allow_origins=["http://localhost:3000"]

@app.get("/")
def root():
    return {"ok": True}

client = TestClient(app)
r = client.get("/", headers={"Origin": "http://localhost:3000"})
print("allow-origin:", r.headers.get("access-control-allow-origin"))


**Your explanation:**

> _Write 2–3 sentences here about what CORS does and why a browser would block a request without it._


## 🎁 Bonus — Yield-Based DB Dependency

**Problem:** Write a yield-based dependency `get_db()` that:
1. Creates a fake DB (a dict like `{"rows": [...]}`)
2. `yield`s it
3. Prints `"closing db"` on teardown

Use it in an endpoint and call it with `TestClient`. You should see `"closing db"` printed after each request.

💡 **Hint:** put the teardown in `finally:` so it runs even if the route raises.


In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient

app = FastAPI()

# TODO: define get_db with yield

# TODO: add an endpoint that uses it

client = TestClient(app)
print(client.get("/data").json())


---

✅ Once all tasks run cleanly, you're done with Day 4. Next up: **auth with API keys & JWT**.
